# 作業 1：多隱藏層全連接神經網路的計算



## 一、準備套件

NumPy 用於矩陣運算。Gradio 用於第二題的互動介面

In [ ]:
%pip -q install "gradio==6.29.1"

In [ ]:
import numpy as np

np.set_printoptions(precision=8, suppress=True)

### 1. 定義激發函數與交叉熵函數


In [ ]:
def relu(z):
    return np.maximum(0, z)


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    result = np.empty_like(z)
    positive = z >= 0
    result[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1.0 + exp_z)
    return result


def softmax(z):
    shifted_z = z - np.max(z)
    exp_z = np.exp(shifted_z)
    return exp_z / np.sum(exp_z)


def cross_entropy_error(y_hat, y):
    return -np.sum(y * np.log(y_hat + 1e-7))


### 2. 輸入資料與各層參數

自行產生具有六個特徵值的一維向量：`x = [1.0, 0.5, -1.0, 2.0, 0.0, 1.5]`。

各層的每一條連線對應一個權重，每一個非輸入層神經元有一個偏值。
因此，一層的參數數量為「輸入神經元數 × 輸出神經元數 + 輸出神經元數」。

In [ ]:
x = np.array([1.0, 0.5, -1.0, 2.0, 0.0, 1.5])
print("x", np.array(x))
print("x" + ".shape =", x.shape)

W1 = np.array([
    [ 0.2, -0.1,  0.3,  0.0,  0.1],
    [ 0.1,  0.4, -0.2,  0.2,  0.0],
    [-0.3,  0.2,  0.1, -0.1,  0.4],
    [ 0.4, -0.2,  0.0,  0.3, -0.1],
    [ 0.0,  0.1,  0.2, -0.3,  0.2],
    [-0.1,  0.3, -0.4,  0.1,  0.3],
])
b1 = np.array([0.1, -0.2, 0.05, 0.0, -0.1])

W2 = np.array([
    [ 0.3, -0.2,  0.1,  0.4],
    [-0.1,  0.5,  0.2, -0.3],
    [ 0.4,  0.1, -0.2,  0.2],
    [ 0.2, -0.3,  0.4,  0.1],
    [-0.2,  0.2,  0.3, -0.4],
])
b2 = np.array([0.05, -0.1, 0.1, -0.05])

W3 = np.array([
    [ 0.2, -0.3,  0.4,  0.1],
    [-0.1,  0.2,  0.3, -0.4],
    [ 0.3,  0.1, -0.2,  0.2],
    [-0.2,  0.4,  0.1,  0.3],
])
b3 = np.array([0.1, -0.1, 0.0, 0.2])

W4 = np.array([
    [ 1.2, -0.4, 0.3],
    [-0.3,  1.3, 0.2],
    [ 0.1, -0.2, 1.0],
    [-0.5,  0.2, 0.7],
])
b4 = np.array([0.1, -0.1, 0.2])

# 列出每一層的權重、偏值、形狀，以及各自的參數數量。
layers = [(W1, b1), (W2, b2), (W3, b3), (W4, b4)]
total_parameters = 0
for i, (W, b) in enumerate(layers, start=1):
    print(f"=== 第 {i} 層 ===")
    show_array(f"W{i}", W)
    show_array(f"b{i}", b)
    count = W.size + b.size
    total_parameters += count
    print(f"W{i} 參數數量 = {W.size}")
    print(f"b{i} 參數數量 = {b.size}")
    print(f"本層參數合計 = {count}\n")

print("全網路參數總數 =", total_parameters)

### 3. 第一隱藏層：5 個神經元，ReLU



In [ ]:
z1 = np.dot(x, W1) + b1
h1 = relu(z1)
print("z1", np.array(z1))
print("z1" + ".shape =", z1.shape)
print("h1", np.array(h1))
print("h1" + ".shape =", h1.shape)

### 4. 第二隱藏層：4 個神經元，ReLU


In [ ]:
z2 = np.dot(h1, W2) + b2
h2 = relu(z2)
print("z2", np.array(z2))
print("z2" + ".shape =", z2.shape)
print("h2", np.array(h2))
print("h2" + ".shape =", h2.shape)

### 5. 第三隱藏層：4 個神經元，Sigmoid


In [ ]:
z3 = np.dot(h2, W3) + b3
h3 = sigmoid(z3)
print("z3", np.array(z3))
print("z3" + ".shape =", z3.shape)
print("h3", np.array(h3))
print("h3" + ".shape =", h3.shape)

### 6. 輸出層：3 個神經元，Softmax

In [ ]:
z4 = np.dot(h3, W4) + b4
y_hat = softmax(z4)
print("z4", np.array(z4))
print("z4" + ".shape =", z4.shape)
print("y_hat", np.array(y_hat))
print("y_hat" + ".shape =", y_hat.shape)

print("機率總和 =", np.sum(y_hat))
for class_id, probability in enumerate(y_hat, start=1):
    print(f"第 {class_id} 類的機率 = {probability:.8f} ({probability:.4%})")
print("預測類別 =", int(np.argmax(y_hat) + 1))

### 7. 以交叉熵衡量誤差


In [ ]:
y = np.array([0, 0, 1])
loss = cross_entropy_error(y_hat, y)
print("y", np.array(y))
print("y" + ".shape =", y.shape)
print(f"cross_entropy_error(y_hat, y) = {loss:.12f}")
print(f"-ln(y_hat[2] + 1e-7) = {-np.log(y_hat[2] + 1e-7):.12f}")

## 二、將神經網路封裝成 dnn(x)

`dnn` 只有一個輸入參數 `x`，回傳三個類別的機率向量 `y_hat`。
直接使用第一題已定義的 `W1` 到 `W4`、`b1` 到 `b4`，函數內不重新產生或更新參數，因此兩題使用的權重與偏值完全相同。

In [ ]:
def dnn(x):
    """輸入六個特徵值的一維向量，回傳三個類別的機率。"""
    x = np.asarray(x, dtype=float)
    if x.shape != (6,):
        raise ValueError("x 必須是具有 6 個特徵值的一維陣列，shape 為 (6,)。")
    if not np.all(np.isfinite(x)):
        raise ValueError("六個特徵值都必須是有限的數字。")

    z1 = np.dot(x, W1) + b1
    h1 = relu(z1)
    z2 = np.dot(h1, W2) + b2
    h2 = relu(z2)
    z3 = np.dot(h2, W3) + b3
    h3 = sigmoid(z3)
    z4 = np.dot(h3, W4) + b4
    y_hat = softmax(z4)
    return y_hat


y_hat_dnn = dnn(x)
print("第一題 y_hat", np.array(y_hat))
print("第一題 y_hat" + ".shape =", y_hat.shape)
print("dnn(x)", np.array(y_hat_dnn))
print("dnn(x)" + ".shape =", y_hat_dnn.shape)
print("逐項差值 =", y_hat_dnn - y_hat)
print("結果是否相同（np.allclose）=", np.allclose(y_hat_dnn, y_hat))

# 驗證題目的核心要求；若結果不符，會立即顯示錯誤。
assert [(W.shape, b.shape) for W, b in layers] == [
    ((6, 5), (5,)), ((5, 4), (4,)), ((4, 4), (4,)), ((4, 3), (3,))
]
assert total_parameters == 94
assert y_hat_dnn.shape == (3,)
assert np.allclose(y_hat_dnn, y_hat)
assert np.isclose(np.sum(y_hat_dnn), 1.0)
assert np.all(y_hat_dnn >= 0)
assert np.isclose(loss, -np.log(y_hat_dnn[2] + 1e-7))
print("形狀、參數總數、機率總和、兩題一致性及交叉熵檢查：全部通過。")

### Gradio 互動介面

介面使用六個 `gr.Number` 輸入欄位。包裝函數 `predict_class` 將六個值組成一維向量，再呼叫 `dnn(x)`。
輸出只顯示機率最高的類別 **1、2 或 3**，符合題目要求。
若機率並列最大，`np.argmax` 會選擇其中編號最小的類別。

下列介面使用固定自訂參數，供觀察神經網路計算，不代表已訓練的分類器。

In [ ]:
import gradio as gr


def predict_class(x1, x2, x3, x4, x5, x6):
    values = [x1, x2, x3, x4, x5, x6]
    if any(value is None for value in values):
        raise gr.Error("請完整輸入六個特徵值。")
    try:
        probabilities = dnn(np.array(values, dtype=float))
    except ValueError as error:
        raise gr.Error(str(error)) from error
    return int(np.argmax(probabilities) + 1)


demo = gr.Interface(
    fn=predict_class,
    inputs=[
        gr.Number(label=f"特徵 x{i + 1}", value=float(value))
        for i, value in enumerate(x)
    ],
    outputs=gr.Number(label="機率最高的類別（1、2 或 3）", precision=0),
    title="作業 1：DNN 分類計算",
    description="輸入六個特徵值，計算 6→5→4→4→3 神經網路的預測類別。",
    examples=[x.tolist(), [0, 0, 0, 0, 0, 0], [-1, 1, 0.5, -0.5, 2, 1]],
    cache_examples=False,
    analytics_enabled=False,
    api_name="predict",
)

print("Gradio 版本：", gr.__version__)
print("介面已建立。")
print("以第一題輸入測試 predict_class：", predict_class(*x.tolist()))

### 啟動介面

In [ ]:
demo.launch(share=True, inline=True)